#Accessing and Downloading NISAR data


Install and import necessary libraries and mount to drive if using Google Colab

In [ ]:
!pip install earthaccess
!pip install xarray

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.8/87.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 74.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.3.0 which is incompatible.


In [ ]:
!pip install rioxarray

In [4]:
import earthaccess
import rioxarray
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
import os
import pyproj
from shapely.ops import transform
from shapely.geometry import Point


/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Define search parameters
For this example, `'NISAR_L2_GCOV_BETA_V1'` Level 2 GCOV data is chosen based on recommended dataset for AGB. To select a different dataset, change the `short_name` variable to a different dataset link.

First, we extract unique coordinate values (longitude, latitude) from a CSV file.


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/AGB_MERGED.csv')


In [ ]:
# get unique longitude and latitude values in df
unique_coordinates = df[['longitude', 'latitude']].drop_duplicates()
unique_coordinates.head()

,longitude,latitude
0,-88.604370,13.227360
87,-88.329576,13.187182
133,-88.336737,13.199314
176,-88.328149,13.202545
255,-88.777010,13.249280


In [ ]:
# convert unique coordinates to input list
coordinates = unique_coordinates.values.tolist()
coordinates

[[-88.60436965931343, 13.227360376425882],
 [-88.32957619089184, 13.18718236595375],
 [-88.33673736866605, 13.199313945612474],
 [-88.32814867984379, 13.202544595221848],
 [-88.77701022951307, 13.249280282257898],
 [-88.78627945640167, 13.253780866694571],
 [-88.45847120700725, 13.216425037256805],
 [-88.45282895239485, 13.222649375363313],
 [-88.39708878488749, 13.192148089259586],
 [-88.5611707304826, 13.25777524858816],
 [-88.56947923249115, 13.240579840183145],
 [-88.81947537256227, 13.25838852729372],
 [-88.829697438107, 13.26988873318014],
 [-88.85941098322873, 13.290139029224513],
 [-88.87336361287223, 13.2969725586972],
 [-88.8811135458455, 13.306055966759253],
 [-88.94430740618901, 13.334139958571113],
 [-88.82980837402025, 13.286222168279906],
 [-88.95555744426943, 13.32530666005413],
 [-88.85333616230449, 13.270388924217926],
 [-88.89086322384209, 13.342278442231413],
 [-88.82673, 13.25972],
 [-88.92136, 13.3442],
 [-88.85316944551337, 13.280111192559334],
 [-88.859002725084

Define search range for NISAR data:

*   `short_name` = name of dataset from NASA Earthdata catalogue
*   `time range` = time range to search for granules
*   `buffer` = buffer in meters around each coordinate. Used to create polygon for each coordinate.
* `count` = number of results to return for each coordinate search

In [ ]:
## USER DEFINED VARIABLES ##
short_name = 'NISAR_L2_GCOV_BETA_V1'
time_range = ('2025-12-01', '2026-03-01')
buffer = 0.001 # 100 meter buffer around each point
count = 1 #limit the search to top 10 results


### Authenticate with Earthdata

Log in to NASA Earthdata with your username and password. This is necessary to access and download the NISAR data.

In [ ]:
earthaccess.login()

### Search for NISAR Granules

First, we iterate through the provided coordinates, create a bounding box (polygon based on the defined buffer) for each coordinate, and then use `earthaccess.search_data` to find relevant NISAR granules with our previously defined search parameters.

The results are then opened and stored in `granules` to make the data accessible for further processing.

In [ ]:
# create buffer polygons for each coordinates and search for data
results = []
for lon, lat in coordinates:

    point_bbox = (lon - buffer, lat - buffer, lon + buffer, lat + buffer)
    result = earthaccess.search_data(
        short_name=short_name,
        temporal=time_range,
        bounding_box=point_bbox,
        cloud_hosted=True,
        count=count
    )
    results.extend(result)

In [ ]:
granules = earthaccess.open(results)

/usr/local/lib/python3.12/dist-packages/earthaccess/store.py:516: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
granules

### Extract and Process Data from Granules

This loop iterates through each opened granule, extracts its metadata (including the `date`), and then processes the data for each coordinate:

1.  **Coordinate Transformation**: Renames coordinates to standard `x`/`y` format and transforms (longitude, latitude) to native Coordinate Reference System (CRS) of NISAR data.
2.  **Value Extraction**: If the coordinate is within bounds, we extract the `HHHH` and `HVHV` polarization values via `method='nearest'` to find the closest data point.
3.  **Data Aggregation**: The extracted date, coordinates, and NISAR values are appended to a list (`all_data`).

Finally, a pandas DataFrame `df` is created from `all_data` and its head is displayed.

In [ ]:
all_data = []

for i, f in enumerate(granules):
    #get metadata and date
    with xr.open_dataset(f, engine='h5netcdf', group='science/LSAR/GCOV/grids/frequencyA',backend_kwargs={'phony_dims': 'sort'}) as ds:

        # conver to correct coordinates to epsg format
        ds = ds.rename({'xCoordinates': 'x', 'yCoordinates': 'y'})

        # conver to native NISAR format
        native_crs = f"EPSG:{ds.projection.epsg_code}"
        ds = ds.rio.write_crs(native_crs)
        transformer = pyproj.Transformer.from_crs("EPSG:4326", native_crs, always_xy=True)

        # get time
        granule_metadata = results[i]["umm"]
        granule_date = granule_metadata['TemporalExtent']['RangeDateTime']['BeginningDateTime']


        for lon, lat in coordinates:
            target_x, target_y = transformer.transform(lon, lat)

            # access data for coordinates only
            if (ds.x.min() <= target_x <= ds.x.max()) and (ds.y.min() <= target_y <= ds.y.max()):
                try:

                    val_hhhh = ds['HHHH'].sel(x=target_x, y=target_y, method='nearest').values.item()
                    val_hvhv = ds['HVHV'].sel(x=target_x, y=target_y, method='nearest').values.item()



                    all_data.append({
                        'date': granule_date,
                        'longitude': lon,
                        'latitude': lat,
                        'nisar_value_hvhv': val_hvhv,
                        'nisar_value_hhhh': val_hhhh
                    })
                except Exception as e:
                    print(f"Point ({lon}, {lat}) not in granule {i}: {e}")

# create df and preview
df = pd.DataFrame(all_data)
df.head()


,date,longitude,latitude,nisar_value_hvhv,nisar_value_hhhh
0,2025-12-13T11:30:58.000000Z,-88.604370,13.227360,0.022314,0.109666
1,2025-12-13T11:30:58.000000Z,-88.329576,13.187182,NaN,NaN
2,2025-12-13T11:30:58.000000Z,-88.336737,13.199314,NaN,NaN
3,2025-12-13T11:30:58.000000Z,-88.328149,13.202545,NaN,NaN
4,2025-12-13T11:30:58.000000Z,-88.777010,13.249280,0.028810,0.045714


In [ ]:
# sort df by longitude
df = df.sort_values(by='longitude', ascending=False)
df

### Identify and Remove Missing Data

Next, we inspect the DataFrame `df` to ensure that only complete data entries are kept for further analysis. We filter the DataFrame `df` to show rows that contain any missing values, and remove all rows that contain any `NaN` values. The cleaned DataFrame is then displayed.

In [ ]:
df_nan = df[df.isna().any(axis=1)]
df_nan.head()

,date,longitude,latitude,nisar_value_hvhv,nisar_value_hhhh
1,2025-12-13T11:30:58.000000Z,-88.329576,13.187182,NaN,NaN
2,2025-12-13T11:30:58.000000Z,-88.336737,13.199314,NaN,NaN
3,2025-12-13T11:30:58.000000Z,-88.328149,13.202545,NaN,NaN


In [ ]:
# drop nulls
df = df.dropna()
df

,date,longitude,latitude,nisar_value_hvhv,nisar_value_hhhh
0,2025-12-13T11:30:58.000000Z,-88.604370,13.227360,0.022314,0.109666
4,2025-12-13T11:30:58.000000Z,-88.777010,13.249280,0.028810,0.045714
5,2025-12-13T11:30:58.000000Z,-88.786279,13.253781,0.020070,0.071964
6,2025-12-13T11:30:58.000000Z,-88.458471,13.216425,0.020845,0.027211
7,2025-12-13T11:30:58.000000Z,-88.452829,13.222649,0.000495,0.001906
...,...,...,...,...,...
121,2025-12-13T11:30:58.000000Z,-48.013617,-0.749333,0.031607,0.086866
122,2025-12-13T11:30:58.000000Z,-48.013550,-0.749483,0.035105,0.149189
123,2025-12-13T11:30:58.000000Z,-48.013333,-0.749567,0.035268,0.157301
124,2025-12-13T11:30:58.000000Z,-48.013217,-0.749733,0.060446,0.210852


### Save Extracted NISAR Data and Coordinates to CSV

The final step is saving the processed and cleaned NISAR data (stored in `df`) to a CSV file in a directory of your choosing (we used Google Drive).

Extracted coordinates (stored in `df_coord`) are also saved to a CSV file (`nisar_coordinates.csv`).

In [ ]:
# save df to csv
filename = "/content/drive/MyDrive/nisar_data.csv"
df.to_csv(filename, index=False)
print(f"Download complete. NISAR data saved to {filename}.")

Download complete. NISAR data saved to /content/drive/MyDrive/nisar_data.csv.


In [ ]:
# save coordinates to csv
df_coord = df[['longitude', 'latitude']]
df_coord.head()


,longitude,latitude
0,-88.604370,13.227360
4,-88.777010,13.249280
5,-88.786279,13.253781
6,-88.458471,13.216425
7,-88.452829,13.222649


In [ ]:
filename = "/content/drive/MyDrive/nisar_coordinates.csv"
df_coord.to_csv(filename, index=False)
print(f"Download complete. NISAR coordinates saved to {filename}.")

Download complete. NISAR coordinates saved to /content/drive/MyDrive/nisar_coordinates.csv.


# Main Function

In [ ]:
if __name__ == "__main__":
    results = []
    for lon, lat in coordinates:

        point_bbox = (lon - buffer, lat - buffer, lon + buffer, lat + buffer)
        result = earthaccess.search_data(
            short_name=short_name,
            temporal=time_range,
            bounding_box=point_bbox,
            cloud_hosted=True,
            count=10
        )
        results.extend(result)
    granules = earthaccess.open(results)
    all_data = []

    for i, f in enumerate(granules):
        # get metadata and date
        with xr.open_dataset(f,
                             engine='h5netcdf',
                             group='science/LSAR/GCOV/grids/frequencyA',
                             backend_kwargs={'phony_dims': 'sort'}) as ds:

            # conver to correct coordinates to epsg format
            ds = ds.rename({'xCoordinates': 'x', 'yCoordinates': 'y'})

            # convert to native NISAR format
            native_crs = f"EPSG:{ds.projection.epsg_code}"
            ds = ds.rio.write_crs(native_crs)
            transformer = pyproj.Transformer.from_crs("EPSG:4326", native_crs, always_xy=True)

            # get time
            granule_metadata = results[i]["umm"]
            granule_date = granule_metadata['TemporalExtent']['RangeDateTime']['BeginningDateTime']


            for lon, lat in coordinates:
                target_x, target_y = transformer.transform(lon, lat)

                # access data for coordinates only
                if (ds.x.min() <= target_x <= ds.x.max()) and (ds.y.min() <= target_y <= ds.y.max()):
                    try:
                        # extracting HHHH and HVHV data elements, but more/differet data can be added here
                        val_hhhh = ds['HHHH'].sel(x=target_x, y=target_y, method='nearest').values.item()
                        val_hvhv = ds['HVHV'].sel(x=target_x, y=target_y, method='nearest').values.item()
                        
                        all_data.append({
                            'date': granule_date,
                            'longitude': lon,
                            'latitude': lat,
                            'nisar_value_hvhv': val_hv,
                            'nisar_value_hhhh': val_hh
                        })
                    except Exception as e:
                        print(f"Point ({lon}, {lat}) not in granule {i}: {e}")

    # create df and save to csv
    df = pd.DataFrame(all_data)
    df.to_csv("nisar_extracted_values.csv", index=False)
    print("Download complete. NISAR data saved to 'nisar_extracted_values.csv'.")

# MERGE NISAR PARAMETERS WITH AGB DATA

In [26]:
NISAR_DATA_CSV = "../../../DATA/AGB_DATA/Merged_Data/NISAR/nisar_data.csv"
nisar_df = pd.read_csv(NISAR_DATA_CSV)
nisar_df.columns

Index(['date', 'longitude', 'latitude', 'nisar_value_hvhv',
       'nisar_value_hhhh'],
      dtype='object')

In [27]:
nisar_df.shape

(123, 5)

In [28]:
AGB_INPUT_CSV  = "../../../DATA/AGB_DATA/Merged_Data/AGB_MERGED.csv"
agb_df = pd.read_csv(AGB_INPUT_CSV)
agb_df.columns

Index(['dataset', 'plot_id', 'start_date', 'end_date', 'latitude', 'longitude',
       'diameter', 'height', 'species', 'plant_AGB_kg'],
      dtype='object')

In [29]:
agb_df.shape

(8774, 10)

## Merge data

In [31]:
# Keep only AGB rows whose coordinates exist in NISAR
filtered_agb_df = agb_df.merge(nisar_df, on=['longitude', 'latitude'], how='inner')

In [32]:
print(f"Original AGB rows:  {len(agb_df)}")
print(f"Filtered AGB rows:  {len(filtered_agb_df)}")
print(filtered_agb_df.columns)

Original AGB rows:  8774
Filtered AGB rows:  3176
Index(['dataset', 'plot_id', 'start_date', 'end_date', 'latitude', 'longitude',
       'diameter', 'height', 'species', 'plant_AGB_kg', 'date',
       'nisar_value_hvhv', 'nisar_value_hhhh'],
      dtype='object')


## SAVE TO FILE

In [33]:
AGB_OUTPUT_CSV = "../../../DATA/AGB_DATA/Merged_Data/NISAR/Nisar_AGB.csv"
filtered_agb_df.to_csv(AGB_OUTPUT_CSV, index=False)